In [16]:
import re
import numpy as np
import pandas as pd

In [17]:
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

### Допоміжні функції

In [18]:
def snake_case(name: str) -> str:
    name = re.sub(r"^\s*\d+\.\s*", "", str(name))
    name = name.strip().lower()
    name = re.sub(r"[^\w\s]", "", name)
    return re.sub(r"\s+", "_", name)

In [19]:
#Обрізати пробіли; порожні/'N/A'/'null' -> NaN

def strip_strings(df: pd.DataFrame) -> pd.DataFrame:
    na_tokens = {"", "nan", "NaN", "N/A", "n/a", "NA", "null", "None", "-"}
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_string_dtype(df[col]):
            s = df[col].astype("string").str.strip()
            df[col] = s.mask(s.isin(na_tokens).fillna(False))
    return df

In [20]:
#Конвертує в число, але не чіпає колонку, якщо конвертація знищує понад max_loss непорожніх значень

def to_numeric(df: pd.DataFrame, cols, report: dict, max_loss=0.2) -> pd.DataFrame:
    for c in cols:
        if c not in df.columns:
            continue
        original = df[c]
        converted = pd.to_numeric(original, errors="coerce")
        n_nonnull = int(original.notna().sum())
        lost = int((converted.isna() & original.notna()).sum())
        if n_nonnull and lost / n_nonnull > max_loss:
            report[f"УВАГА {c}: не числова колонка, конвертацію пропущено"] = (
                f"{lost}/{n_nonnull} значень не є числами; приклади: "
                f"{original.dropna().unique()[:5].tolist()}"
            )
            continue
        df[c] = converted
    return df

In [21]:
def apply_range(df: pd.DataFrame, rules: dict, report: dict) -> pd.DataFrame:
    for col, (lo, hi) in rules.items():
        if col not in df.columns or not pd.api.types.is_numeric_dtype(df[col]):
            continue
        bad = (df[col] < lo) | (df[col] > hi)
        if bad.sum():
            report[f"{col}: поза діапазоном [{lo}, {hi}]"] = int(bad.sum())
        df.loc[bad, col] = np.nan
    return df

In [22]:
def cap_outliers_iqr(df: pd.DataFrame, cols, k=3.0, report=None) -> pd.DataFrame:
    for col in cols:
        if col not in df.columns or not pd.api.types.is_numeric_dtype(df[col]) \
                or pd.api.types.is_bool_dtype(df[col]):
            continue
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        if iqr == 0:
            continue
        lo, hi = q1 - k * iqr, q3 + k * iqr
        mask = (df[col] < lo) | (df[col] > hi)
        if mask.sum() and report is not None:
            report[f"{col}: IQR-викиди (обрізано)"] = int(mask.sum())
        df[col] = df[col].clip(lo, hi)
    return df

In [23]:
#protect - колонки, які не видаляємо і не заповнюємо (ID)
#unknown_cols - категоріальні колонки, де пропуск = 'Unknown'

def impute(df: pd.DataFrame, report: dict, max_missing_frac=0.5,
           protect=(), unknown_cols=()) -> pd.DataFrame:
    too_empty = [c for c in df.columns
                 if c not in protect and df[c].isna().mean() > max_missing_frac]
    if too_empty:
        report["Видалено колонки (>50% пропусків)"] = too_empty
        df = df.drop(columns=too_empty)

    for col in df.columns:
        if col in protect:
            continue
        n_missing = int(df[col].isna().sum())
        if n_missing == 0:
            continue
        if col in unknown_cols:
            df[col] = df[col].fillna("Unknown")
        elif pd.api.types.is_bool_dtype(df[col]):
            df[col] = df[col].fillna(df[col].mode(dropna=True).iloc[0])
        elif pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(df[col].median())
        else:
            mode = df[col].mode(dropna=True)
            df[col] = df[col].fillna(mode.iloc[0] if len(mode) else "Unknown")
        report[f"{col}: заповнено пропусків"] = n_missing
    return df

In [24]:
def to_bool(series: pd.Series) -> pd.Series:
    mapping = {"yes": True, "y": True, "true": True, "1": True, "1.0": True,
               "no": False, "n": False, "false": False, "0": False, "0.0": False}
    return series.astype("string").str.lower().map(mapping).astype("boolean")

In [25]:
def standardize_categories(series: pd.Series, mapping: dict = None) -> pd.Series:
    s = series.astype("string").str.strip().str.replace(r"\s+", " ", regex=True).str.title()
    if mapping:
        s = s.replace(mapping)
    return s

In [26]:
def print_report(name, before, after, report):
    print(f"\nЗВІТ ОЧИЩЕННЯ: {name}")
    print(f"Розмір до:    {before}")
    print(f"Розмір після: {after}")
    for k, v in report.items():
        print(f" - {k}: {v}")

## 1. sleep_doomscrolling_habits.csv

In [27]:
def clean_sleep_doomscrolling(path="sleep_doomscrolling_habits.csv",
                              out="sleep_doomscrolling_habits_clean.csv"):
    df = pd.read_csv(path)
    before, report = df.shape, {}

    df.columns = [snake_case(c) for c in df.columns]

    junk = [c for c in df.columns if re.fullmatch(r"d\d+", c) or c.startswith("unnamed")]
    if junk:
        df = df.drop(columns=junk)
        report["Видалено службові колонки"] = junk

    df = strip_strings(df)

    numeric_cols = [
        "age", "bedtime_screen_time_minutes", "total_daily_screen_time_hours",
        "doomscroll_sessions_per_night", "avg_doomscroll_session_minutes",
        "sleep_hours_per_night", "sleep_latency_minutes", "number_of_night_wakeups",
        "caffeine_intake_mg_per_day", "anxiety_score", "stress_score",
        "sleep_quality_score", "daytime_fatigue_score", "number_of_news_apps_used",
        "phone_checks_per_night", "exercise_minutes_per_day",
        "days_since_last_digital_detox", "weekly_sleep_debt_hours",
    ]
    df = to_numeric(df, numeric_cols, report)   # respondent_id не конвертуємо

    # Дублікати: за ID - лише де ID заповнений
    if "respondent_id" in df.columns:
        dup_id = df.duplicated(subset="respondent_id", keep="first") & df["respondent_id"].notna()
        report["Дублікати за respondent_id"] = int(dup_id.sum())
        df = df[~dup_id]
    n = int(df.duplicated().sum())
    df = df.drop_duplicates()
    report["Повні дублікати"] = n

    ranges = {
        "age": (10, 100),
        "bedtime_screen_time_minutes": (0, 600),
        "total_daily_screen_time_hours": (0, 24),
        "doomscroll_sessions_per_night": (0, 50),
        "avg_doomscroll_session_minutes": (0, 480),
        "sleep_hours_per_night": (0, 24),
        "sleep_latency_minutes": (0, 480),
        "number_of_night_wakeups": (0, 30),
        "caffeine_intake_mg_per_day": (0, 1500),
        "anxiety_score": (0, 10),
        "stress_score": (0, 10),
        "sleep_quality_score": (0, 10),
        "daytime_fatigue_score": (0, 10),
        "number_of_news_apps_used": (0, 50),
        "phone_checks_per_night": (0, 200),
        "exercise_minutes_per_day": (0, 600),
        "days_since_last_digital_detox": (0, 3650),
        "weekly_sleep_debt_hours": (-50, 100),
    }
    df = apply_range(df, ranges, report)

    for c in ["gender", "occupation_status", "country_region",
              "primary_device_used_at_night", "bedtime_routine_type",
              "sleep_quality_category"]:
        if c in df.columns:
            df[c] = standardize_categories(df[c])

    for c in ["uses_night_mode", "keeps_phone_in_bedroom",
              "consumes_negative_news_content", "uses_sleep_tracking_app", "doomscroller"]:
        if c in df.columns:
            df[c] = to_bool(df[c])

    if {"doomscroll_sessions_per_night", "avg_doomscroll_session_minutes"} <= set(df.columns):
        total = df["doomscroll_sessions_per_night"] * df["avg_doomscroll_session_minutes"]
        bad = total > 12 * 60
        report["Doomscrolling за ніч >12 год"] = int(bad.sum())
        df.loc[bad, ["doomscroll_sessions_per_night", "avg_doomscroll_session_minutes"]] = np.nan

    if {"sleep_hours_per_night", "total_daily_screen_time_hours"} <= set(df.columns):
        bad = (df["sleep_hours_per_night"] + df["total_daily_screen_time_hours"]) > 30
        report["Сон + екранний час > 30 год"] = int(bad.sum())
        df.loc[bad, "total_daily_screen_time_hours"] = np.nan

    df = impute(df, report, protect=("respondent_id",))
    df = cap_outliers_iqr(df, numeric_cols, report=report)

    df.to_csv(out, index=False)
    print_report("sleep_doomscrolling_habits", before, df.shape, report)
    return df

## 2. smmh.csv

In [28]:
def clean_smmh(path="smmh.csv", out="smmh_clean.csv"):
    df = pd.read_csv(path)
    before, report = df.shape, {}

    short_names = [
        "timestamp", "age", "gender", "relationship_status", "occupation_status",
        "organization_type", "uses_social_media", "platforms",
        "avg_daily_time_social_media", "use_without_purpose", "distracted_when_busy",
        "restless_without_sm", "easily_distracted", "bothered_by_worries",
        "difficulty_concentrating", "compare_to_others", "feeling_about_comparisons",
        "seek_validation", "feel_depressed", "interest_fluctuation", "sleep_issues",
    ]
    if len(df.columns) == len(short_names):
        df.columns = short_names
    else:
        df.columns = [snake_case(c) for c in df.columns]
        report["Увага"] = "кількість колонок відрізняється, використано snake_case"

    df = strip_strings(df)

    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

    df = to_numeric(df, ["age"], report)
    df = apply_range(df, {"age": (10, 100)}, report)

    likert = ["easily_distracted", "bothered_by_worries", "difficulty_concentrating",
              "compare_to_others", "feeling_about_comparisons", "seek_validation",
              "feel_depressed", "interest_fluctuation", "sleep_issues",
              "use_without_purpose", "distracted_when_busy", "restless_without_sm"]
    df = to_numeric(df, likert, report)
    df = apply_range(df, {c: (1, 5) for c in likert}, report)

    n = int(df.duplicated().sum())
    df = df.drop_duplicates()
    report["Повні дублікати"] = n

    if "gender" in df.columns:
        g = df["gender"].astype("string").str.strip().str.lower()
        g = g.replace({"male": "Male", "m": "Male", "man": "Male",
                       "female": "Female", "f": "Female", "woman": "Female"})
        df["gender"] = g.where(g.isin(["Male", "Female"]) | g.isna(), "Other")

    for c in ["relationship_status", "occupation_status", "organization_type"]:
        if c in df.columns:
            df[c] = standardize_categories(df[c])

    if "uses_social_media" in df.columns:
        df["uses_social_media"] = to_bool(df["uses_social_media"])

    if "platforms" in df.columns:
        plat = (df["platforms"].astype("string").str.split(r"\s*,\s*", regex=True)
                .apply(lambda x: [p.strip().title() for p in x] if isinstance(x, list) else []))
        all_platforms = sorted({p for lst in plat for p in lst if p})
        for p in all_platforms:
            col = "platform_" + re.sub(r"[^a-z0-9]+", "_", p.lower()).strip("_")
            df[col] = plat.apply(lambda lst, p=p: int(p in lst))
        report["One-hot колонок для платформ"] = len(all_platforms)

    if "avg_daily_time_social_media" in df.columns:
        def parse_time(x):
            if pd.isna(x):
                return np.nan
            s = str(x).lower()
            nums = [float(n) for n in re.findall(r"\d+\.?\d*", s)]
            if "less" in s:                       # "Less than an Hour" -> 0.5
                return nums[0] / 2 if nums else 0.5
            if "more" in s and nums:              # "More than 5 hours" -> 6
                return nums[0] + 1
            if len(nums) >= 2:                    # "Between 2 and 3 hours" -> 2.5
                return (nums[0] + nums[1]) / 2
            return nums[0] if nums else np.nan

        df["avg_daily_hours_sm"] = df["avg_daily_time_social_media"].apply(parse_time)
        unparsed = df.loc[df["avg_daily_hours_sm"].isna()
                          & df["avg_daily_time_social_media"].notna(),
                          "avg_daily_time_social_media"].unique()
        if len(unparsed):
            report["УВАГА: не розпізнано значення часу"] = unparsed.tolist()

    # NaN в organization_type = не входить в жодну організацію
    df = impute(df, report, unknown_cols=("organization_type",))

    df.to_csv(out, index=False)
    print_report("smmh", before, df.shape, report)
    return df


## 3. Student Social Media And Mental Health Impact.csv

In [29]:
def clean_student_sm(path="Student Social Media And Mental Health Impact.csv",
                     out="student_sm_clean.csv"):
    df = pd.read_csv(path)
    before, report = df.shape, {}

    df.columns = [snake_case(c) for c in df.columns]
    df = strip_strings(df)

    numeric_cols = ["age", "avg_daily_usage_hours", "daily_unlocks", "study_hours",
                    "physical_activity_hours", "sleep_hours_per_night",
                    "stress_level", "mental_health_score"]
    df = to_numeric(df, numeric_cols, report)

    # stress_level може бути текстовим (Low/Medium/High)
    if "stress_level" in df.columns and not pd.api.types.is_numeric_dtype(df["stress_level"]):
        df["stress_level"] = standardize_categories(df["stress_level"])
        report["stress_level: унікальні значення"] = sorted(
            df["stress_level"].dropna().unique().tolist())
        df["stress_level_num"] = df["stress_level"].map({"Low": 1, "Medium": 2, "High": 3})

    n = int(df.duplicated().sum())
    df = df.drop_duplicates()
    report["Повні дублікати"] = n

    ranges = {
        "age": (14, 60),
        "avg_daily_usage_hours": (0, 24),
        "daily_unlocks": (0, 500),
        "study_hours": (0, 16),
        "physical_activity_hours": (0, 12),
        "sleep_hours_per_night": (2, 14),
        "stress_level": (1, 10),          # застосується лише якщо колонка числова
        "mental_health_score": (0, 100),
    }
    df = apply_range(df, ranges, report)

    day_cols = [c for c in ["avg_daily_usage_hours", "study_hours",
                            "physical_activity_hours", "sleep_hours_per_night"]
                if c in df.columns]
    bad = df[day_cols].sum(axis=1, skipna=True) > 30
    report["Сума годин на добу > 30"] = int(bad.sum())
    df = df[~bad].copy()

    for c in ["gender", "country", "academic_level", "most_used_platform", "purpose_of_use"]:
        if c in df.columns:
            df[c] = standardize_categories(df[c])

    if "most_used_platform" in df.columns:
        df["most_used_platform"] = df["most_used_platform"].replace({
            "Insta": "Instagram", "Ig": "Instagram", "Fb": "Facebook",
            "Yt": "Youtube", "X": "Twitter", "Tik Tok": "Tiktok"})
    if "country" in df.columns:
        df["country"] = df["country"].replace({
            "Usa": "United States", "Us": "United States",
            "United States Of America": "United States", "Uk": "United Kingdom"})

    df = impute(df, report)
    df = cap_outliers_iqr(df, numeric_cols, report=report)

    df.to_csv(out, index=False)
    print_report("Student Social Media And Mental Health Impact", before, df.shape, report)
    return df

## Результати

In [30]:
if __name__ == "__main__":
    df1 = clean_sleep_doomscrolling()
    df2 = clean_smmh()
    df3 = clean_student_sm()


ЗВІТ ОЧИЩЕННЯ: sleep_doomscrolling_habits
Розмір до:    (1000, 30)
Розмір після: (1000, 30)
 - Дублікати за respondent_id: 0
 - Повні дублікати: 0
 - Doomscrolling за ніч >12 год: 0
 - Сон + екранний час > 30 год: 0
 - caffeine_intake_mg_per_day: заповнено пропусків: 50
 - sleep_quality_score: заповнено пропусків: 40
 - exercise_minutes_per_day: заповнено пропусків: 50
 - occupation_status: заповнено пропусків: 30
 - primary_device_used_at_night: заповнено пропусків: 30
 - bedtime_routine_type: заповнено пропусків: 20
 - bedtime_screen_time_minutes: IQR-викиди (обрізано): 3
 - total_daily_screen_time_hours: IQR-викиди (обрізано): 1
 - doomscroll_sessions_per_night: IQR-викиди (обрізано): 1
 - avg_doomscroll_session_minutes: IQR-викиди (обрізано): 4
 - number_of_night_wakeups: IQR-викиди (обрізано): 2
 - caffeine_intake_mg_per_day: IQR-викиди (обрізано): 2
 - days_since_last_digital_detox: IQR-викиди (обрізано): 8

ЗВІТ ОЧИЩЕННЯ: smmh
Розмір до:    (481, 21)
Розмір після: (481, 31)
 - 